## Translation quality audit

Three deliverables:
1. **Back-translation BLEU + chrF.** Sample 200 records per dataset, translate the Croatian back to English with Google NMT, compute BLEU and chrF against the original English using `sacrebleu`. Report overall and per-PR-category. This is the standard MT quality proxy in cross-lingual NLP literature.
2. **Distortion-marker preservation audit.** Build Croatian marker dictionaries empirically by reading actual translations per category, then measure per-category preservation rate on the full PR corpus.
3. **Manual spot-check sampler.** Produce a 30-record CSV stratified across categories, ready to be read and scored 1-5 manually. Aggregate scoring goes into the methodology section.

API cost: back-translation sends ~190k characters total, which fits inside the always-free 500k/month quota. No charge against the trial credit.

Prerequisites: `00_07_translate_en_to_hr.ipynb` finished successfully and the bilingual CSVs exist under `processed/`. `translator.py` is in the same directory. `GOOGLE_TRANSLATE_API_KEY` is set.

## 1. Imports and configuration

In [1]:
import os
import re
import sys
import random
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path.cwd()))
import translator

PROCESSED_DIR = Path("./processed")
CACHE_DIR = PROCESSED_DIR / "translation_cache"
AUDIT_DIR = PROCESSED_DIR / "translation_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TQA_BI = {
    "train": PROCESSED_DIR / "therapistqa_bilingual_train.csv",
    "val":   PROCESSED_DIR / "therapistqa_bilingual_val.csv",
    "test":  PROCESSED_DIR / "therapistqa_bilingual_test.csv",
}
PR_BI = {
    "train": PROCESSED_DIR / "patternreframe_bilingual_train.csv",
    "val":   PROCESSED_DIR / "patternreframe_bilingual_val.csv",
    "test":  PROCESSED_DIR / "patternreframe_bilingual_test.csv",
}
CR_BI = PROCESSED_DIR / "cognitive_reframing_bilingual.csv"

PR_CATEGORIES = [
    "catastrophizing", "discounting_positive", "labeling",
    "mental_filtering", "mind_reading", "fortune_telling",
    "overgeneralization", "personalization",
    "black_and_white", "should_statements",
]
PR_MARKER_COLS = [f"marked_{c}" for c in PR_CATEGORIES]

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Install `sacrebleu` if needed

In [2]:
try:
    import sacrebleu
    print(f"sacrebleu {sacrebleu.__version__} available")
except ImportError:
    print("Installing sacrebleu...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sacrebleu"])
    import sacrebleu
    print(f"sacrebleu {sacrebleu.__version__} installed")

sacrebleu 2.6.0 available


# Part 1 — Back-translation BLEU + chrF

Method: take a sample of EN→HR translated records, translate the HR back to EN with the same NMT system, and compare the round-tripped EN to the original EN using BLEU and chrF. High back-translation scores indicate that the forward translation preserved enough meaning for a reverse pass to reconstruct the original. This is a well-established proxy for forward-translation quality in MT evaluation literature.

Caveats made explicit in Chapter 3.4: back-translation does not measure adequacy directly (a poor forward translation can sometimes round-trip well by chance), and using the same MT system in both directions risks circular flattery (the system's biases cancel out). Still, in the absence of human-rated references, back-translation BLEU/chrF is the conventional sanity check and is what the thesis will report alongside the manual spot-check (Part 3) which addresses these caveats.

## 3. Build back-translation sample (200 per dataset)

For PatternReframe, stratify across the 10 distortion categories (20 per category) so per-category back-translation BLEU is computable. For TQA and CR, simple random sample.

In [3]:
def sample_tqa_for_backtranslation(n=200):
    df = pd.read_csv(TQA_BI["test"])
    df = df.dropna(subset=["patient_question", "patient_question_hr"])
    sample = df.sample(min(n, len(df)), random_state=RANDOM_SEED)
    return sample[["patient_question", "patient_question_hr"]].rename(
        columns={"patient_question": "en", "patient_question_hr": "hr"}
    ).assign(dataset="TQA", category="")

def sample_pr_for_backtranslation(per_category=20):
    df = pd.read_csv(PR_BI["test"]).dropna(subset=["thought", "thought_hr"])
    rows = []
    for cat in PR_CATEGORIES:
        marker_col = f"marked_{cat}"
        sub = df[df[marker_col] > 0]
        if len(sub) == 0:
            continue
        s = sub.sample(min(per_category, len(sub)), random_state=RANDOM_SEED)
        for _, r in s.iterrows():
            rows.append({"en": r["thought"], "hr": r["thought_hr"], "dataset": "PR", "category": cat})
    return pd.DataFrame(rows)

def sample_cr_for_backtranslation(n=200):
    df = pd.read_csv(CR_BI).dropna(subset=["thought", "thought_hr"])
    sample = df.sample(min(n, len(df)), random_state=RANDOM_SEED)
    return sample[["thought", "thought_hr"]].rename(
        columns={"thought": "en", "thought_hr": "hr"}
    ).assign(dataset="CR", category="")

bt_sample = pd.concat([
    sample_tqa_for_backtranslation(200),
    sample_pr_for_backtranslation(20),
    sample_cr_for_backtranslation(200),
], ignore_index=True)

print(f"Back-translation sample: {len(bt_sample):,} records")
print(bt_sample["dataset"].value_counts())
print()
print("Total HR characters to back-translate:", int(bt_sample["hr"].str.len().sum()))

Back-translation sample: 600 records
dataset
TQA    200
PR     200
CR     200
Name: count, dtype: int64

Total HR characters to back-translate: 189500


## 4. Back-translate HR → EN

Uses the same `translator.translate_strings` machinery as Phase D, but with source=hr and target=en. Caches into a dedicated file.

In [4]:
BT_CACHE = CACHE_DIR / "backtranslation_hr_to_en.json"

hr_strings = bt_sample["hr"].tolist()
print("Translating HR → EN...")
bt_map = translator.translate_strings(
    strings=hr_strings,
    cache_path=BT_CACHE,
    source="hr",
    target="en",
    batch_size=50,
)
bt_sample["en_back"] = bt_sample["hr"].map(bt_map)
n_back = bt_sample["en_back"].notna().sum()
print(f"\nBack-translated {n_back}/{len(bt_sample)} records")

Translating HR → EN...
  cache: 0 entries already present
  inputs: 552 unique non-empty strings
  to translate: 552 strings (0 cache hits)
  total characters to send: 187,876 (~$3.76 at $20/1M)
  batch   10/12  (500/552)  elapsed 5s  eta 1s
  done in 6s; cache now has 552 entries

Back-translated 600/600 records


## 5. Compute BLEU and chrF

Per-record scores (sentence-level BLEU and chrF) plus corpus-level scores per dataset and per PR category. Per-record scores let us see whether degradation is uniform or has a long tail of bad cases; corpus-level is the headline number.

In [5]:
import sacrebleu

def sentence_bleu(hyp, ref):
    if not isinstance(hyp, str) or not isinstance(ref, str) or not hyp.strip() or not ref.strip():
        return float("nan")
    return sacrebleu.sentence_bleu(hyp, [ref]).score

def sentence_chrf(hyp, ref):
    if not isinstance(hyp, str) or not isinstance(ref, str) or not hyp.strip() or not ref.strip():
        return float("nan")
    return sacrebleu.sentence_chrf(hyp, [ref]).score

bt_sample["sent_bleu"] = bt_sample.apply(
    lambda r: sentence_bleu(r["en_back"], r["en"]), axis=1
)
bt_sample["sent_chrf"] = bt_sample.apply(
    lambda r: sentence_chrf(r["en_back"], r["en"]), axis=1
)

In [6]:
def corpus_bleu_chrf(refs, hyps):
    pairs = [(h, r) for h, r in zip(hyps, refs)
             if isinstance(h, str) and h.strip() and isinstance(r, str) and r.strip()]
    if not pairs:
        return float("nan"), float("nan"), 0
    hyps_clean, refs_clean = zip(*pairs)
    bleu = sacrebleu.corpus_bleu(list(hyps_clean), [list(refs_clean)]).score
    chrf = sacrebleu.corpus_chrf(list(hyps_clean), [list(refs_clean)]).score
    return bleu, chrf, len(pairs)

# Per-dataset corpus scores.
print("Corpus-level BLEU / chrF, by dataset:")
print("-" * 60)
ds_rows = []
for ds in ["TQA", "PR", "CR"]:
    sub = bt_sample[bt_sample["dataset"] == ds]
    bleu, chrf, n = corpus_bleu_chrf(sub["en"].tolist(), sub["en_back"].tolist())
    ds_rows.append({"dataset": ds, "n": n, "bleu": bleu, "chrf": chrf})
    print(f"  {ds}:  n={n:>3}   BLEU={bleu:>5.2f}   chrF={chrf:>5.2f}")

per_dataset_bleu = pd.DataFrame(ds_rows)

# Per-PR-category corpus scores.
print("\nCorpus-level BLEU / chrF, by PR distortion category:")
print("-" * 60)
cat_rows = []
pr_sub = bt_sample[bt_sample["dataset"] == "PR"]
for cat in PR_CATEGORIES:
    sub = pr_sub[pr_sub["category"] == cat]
    bleu, chrf, n = corpus_bleu_chrf(sub["en"].tolist(), sub["en_back"].tolist())
    cat_rows.append({"category": cat, "n": n, "bleu": bleu, "chrf": chrf})
    print(f"  {cat:<25} n={n:>3}   BLEU={bleu:>5.2f}   chrF={chrf:>5.2f}")

per_category_bleu = pd.DataFrame(cat_rows)

Corpus-level BLEU / chrF, by dataset:
------------------------------------------------------------
  TQA:  n=200   BLEU=49.07   chrF=72.18
  PR:  n=200   BLEU=57.21   chrF=75.42
  CR:  n=200   BLEU=52.21   chrF=74.94

Corpus-level BLEU / chrF, by PR distortion category:
------------------------------------------------------------
  catastrophizing           n= 20   BLEU=55.45   chrF=75.23
  discounting_positive      n= 20   BLEU=62.09   chrF=79.25
  labeling                  n= 20   BLEU=49.29   chrF=71.49
  mental_filtering          n= 20   BLEU=58.59   chrF=75.05
  mind_reading              n= 20   BLEU=64.70   chrF=79.96
  fortune_telling           n= 20   BLEU=50.99   chrF=69.79
  overgeneralization        n= 20   BLEU=57.10   chrF=76.37
  personalization           n= 20   BLEU=48.58   chrF=70.18
  black_and_white           n= 20   BLEU=63.35   chrF=80.16
  should_statements         n= 20   BLEU=60.68   chrF=77.70


In [7]:
# Save back-translation outputs.
bt_sample.to_csv(AUDIT_DIR / "backtranslation_sample.csv", index=False)
per_dataset_bleu.to_csv(AUDIT_DIR / "backtranslation_per_dataset.csv", index=False)
per_category_bleu.to_csv(AUDIT_DIR / "backtranslation_per_pr_category.csv", index=False)
print("Back-translation results written to processed/translation_audit/")

Back-translation results written to processed/translation_audit/


# Part 2 — Distortion-marker preservation audit

The central question for Chapter 4.4: do some distortion categories survive translation better than others, and does the per-category F1 degradation correlate with marker loss?

Method:
1. **Build Croatian marker dictionaries empirically** (Section 6). For each PR category, sample ~50 translated thoughts from the test set, read them, and extract the Croatian lexical patterns that recur for that category. Encoded as regex.
2. **Apply dictionaries to the full PR test set** (Section 7). For each thought labelled with category C, check whether category C's marker pattern matches. Compute per-category preservation rate.
3. **Save the dictionaries** so they can be cited verbatim in Chapter 3.4.

## 6. Marker dictionaries

Built by inspection of translated PR thoughts. Each pattern is a regex; matches are case-insensitive and substring-based. The patterns favour high recall over high precision — for the preservation audit, a false positive is less harmful than a false negative (a missed marker incorrectly reads as translation degradation). The vocabulary below covers common Croatian renderings and includes both masculine and feminine inflections where relevant.

After the first run of Section 7, examine categories with surprisingly low preservation rates and use Section 8's inspection cell to read sample translations. If real markers are being missed, extend the regex below and re-run. Conversely, if a category has surprisingly high preservation, check whether the marker is too generic (e.g. `samo` = 'just/only' is very common in Croatian and matches many things).

In [16]:
# Croatian distortion-marker regexes — initial draft.
# Iterate by reading per-category samples in Section 8.
CROATIAN_MARKERS = {
    "should_statements": (
        # Modal verbs: treba (should/need), morati (must) — every inflection.
        # 'trebao bih', 'trebala bi', 'morao bi', etc. caught by stems.
        r"(treba|trebam|trebaš|trebamo|trebate|trebao|trebala|trebali|trebale|trebalo"
        r"|mora|moram|moraš|moramo|morate|morao|morala|morali|morale|moralo|moraju"
        r"|bi trebao|bi trebala|bi morao|bi morala)"
    ),

    "black_and_white": (
        # Universal/absolute quantifiers + extreme intensifiers.
        # NB: 'samo' (just/only) is a common word; can cause false positives,
        # but mostly correctly indicates restrictive thinking when present.
        r"(nikad|nikada|uvijek|svatko|nitko|ništa|nimalo"
        r"|\bsvi |\bsve |\bsamo |\bjedino |\bjedini |\bjedina "
        r"|potpuno|posve|isključivo|savršen|katastrofaln|totaln|apsolutn|zauvijek)"
    ),

    "catastrophizing": (
        # Disaster vocabulary + dire future-tense predictions ("never will",
        # "will die", "will collapse"). Stem 'propa' catches propasti/propao/
        # propala/propala/propast će/propala je. 'umrij' catches all forms of
        # "die". 'uništ' catches "destroy" inflections.
        r"(propast|propa[oas]|propad|umrij|umro|umrla|poginu"
        r"|katastrof|najgor|strašn|užasn|grozn|smak svijeta|kraj svijeta"
        r"|uništ|razor|smrt"
        r"|nikad neć|nikada neć|zauvijek"
        r"|past ću|past će|past ćemo)"
    ),

    "mind_reading": (
        # Verbs of inferring another's thoughts / certainty about another's state.
        r"(mora da misli|sigurno misli|sigurn[oa] sam da|"
        r"znam što (misl|želi|hoć)|misli da|misle da|mora da je"
        r"|očito|očit[oa] je|kladim se|kladim se da"
        r"|vjerojatno misl|smatra(ju)? da"
        r"|ne voli me|ne sviđam mu se|ne sviđam joj se|ne sviđam im se"
        r"|odluč(io|ila|ili|ile) (je |da )(me |više |ne ))"
    ),

    "fortune_telling": (
        # Future-tense negative predictions. "Neće" + verb is the workhorse.
        # 'nikad neć' often co-occurs with both fortune-telling and
        # catastrophizing (these categories overlap by construction).
        r"(\bneć[uešmo]|\bneće\b|\bnećete|\bnećemo"
        r"|nikad neć|nikada neć|više neć|više neću|više neće"
        r"|past ću|past će|propa(st|d|o|la)"
        r"|izgubit ću|izgubit će|ostat ću sam|zauvijek)"
    ),

    "overgeneralization": (
        # Universal frequency / always-never. Heavy overlap with black_and_white.
        r"(uvijek|nikad|nikada|svaki put|stalno|kontinuirano"
        r"|opet|ponovno|neprestano"
        r"|svi |svatko|nitko|svaki |svaka |svako "
        r"|baš svaki|baš svaka)"
    ),

    "personalization": (
        # Self-blame attributions. Most patterns include "ja", "moj-", "kriv-".
        r"(moja krivnj|krivnja je moja|moja greška|moja je greška"
        r"|kriv sam|kriva sam|kriv je zbog mene|kriva sam ja"
        r"|zbog mene|sve sam ja|sve je do mene"
        r"|odgovoran sam|odgovorna sam|moja odgovornost"
        r"|jer sam ja|to sam ja|ja sam ta|ja sam taj"
        r"|sve sam zaribao|sve sam pokvario|sve sam pokvarila)"
    ),

    "labeling": (
        # Negative trait nouns + "I am X" identity statements.
        # 'propao sam', 'propala sam' = "I'm a failure" — also catastrophizing.
        r"(idiot|glup|bezvrijed|nesposob|gubitnik|propalica|čudak"
        r"|bezna[dn]|jadan|jadna|jadno|beskoristan|beskorisna|beskorisno"
        r"|grozan|grozna|grozno|užasan|užasna|nikakv|sebičn|lijen"
        r"|kukavica|promašaj|promašen"
        r"|loš sam|loša sam|loš[ae] u "
        r"|propao sam|propala sam"
        r"|takv[ai]? sam|nikada neću biti dobar|nikada neću biti dobra)"
    ),

    "mental_filtering": (
        # Mental filter is conceptually negative-only attention. Hard to capture
        # lexically because the signal is what's NOT mentioned. Best proxies:
        # contrastive 'but'/'međutim' constructions and exclusive focus markers.
        r"(jedino vid|jedino što vid|samo vid|samo razmišljam o|samo mogu mislit"
        r"|samo se sjeć|sve negativn"
        r"|usredoto|fokusira"
        r"|\bali \b|međutim|no jer|samo na|samo o |samo u )"
    ),

    "discounting_positive": (
        # 'But/however' constructions + 'doesn't count' + 'just luck' patterns.
        r"(\bali \b|međutim|no jer"
        r"|svejedno|nije važno|ne vrijedi|ne računa|ne broji"
        r"|obična sreć|samo sreć|sreća"
        r"|bilo tko može|svatko može|nije ništa posebno|nije nešto"
        r"|nije zaslug|nezasluženo|slučajno)"
    ),
}

print("Croatian marker dictionaries defined for", len(CROATIAN_MARKERS), "categories.")
for k, v in CROATIAN_MARKERS.items():
    print(f"  {k:<25} regex length: {len(v)} chars")

Croatian marker dictionaries defined for 10 categories.
  should_statements         regex length: 188 chars
  black_and_white           regex length: 170 chars
  catastrophizing           regex length: 186 chars
  mind_reading              regex length: 280 chars
  fortune_telling           regex length: 168 chars
  overgeneralization        regex length: 134 chars
  personalization           regex length: 280 chars
  labeling                  regex length: 311 chars
  mental_filtering          regex length: 163 chars
  discounting_positive      regex length: 192 chars


## 7. Apply markers across the full PR test set

For each thought in the PR test bilingual file, find which categories it is marked with, and for each such category check whether that category's Croatian marker regex matches the Croatian translation. Report per-category preservation rate (hits / total).

In [17]:
pr_test = pd.read_csv(PR_BI["test"]).dropna(subset=["thought", "thought_hr"])
print(f"PR test bilingual: {len(pr_test):,} rows")

per_cat_total = {c: 0 for c in PR_CATEGORIES}
per_cat_hit   = {c: 0 for c in PR_CATEGORIES}
per_cat_compiled = {c: re.compile(p, re.IGNORECASE) for c, p in CROATIAN_MARKERS.items()}

for _, row in pr_test.iterrows():
    hr = str(row["thought_hr"])
    for cat in PR_CATEGORIES:
        if row.get(f"marked_{cat}", 0) > 0:
            per_cat_total[cat] += 1
            if per_cat_compiled[cat].search(hr):
                per_cat_hit[cat] += 1

marker_rows = []
for cat in PR_CATEGORIES:
    total = per_cat_total[cat]
    hit = per_cat_hit[cat]
    rate = hit / total if total else float("nan")
    marker_rows.append({"category": cat, "n": total, "hits": hit, "preservation_rate": rate})

marker_df = pd.DataFrame(marker_rows).sort_values("preservation_rate", ascending=False)
print("\nMarker preservation rate, PR test set (full corpus):")
print("-" * 70)
for _, r in marker_df.iterrows():
    pct = f"{100*r['preservation_rate']:.1f}%" if pd.notna(r['preservation_rate']) else "n/a"
    print(f"  {r['category']:<25} {r['hits']:>5}/{r['n']:<5}  {pct:>7}")

marker_df.to_csv(AUDIT_DIR / "marker_preservation_pr_test.csv", index=False)

# Also save the dictionary itself for citation in Chapter 3.4.
import json
with open(AUDIT_DIR / "croatian_marker_dictionaries.json", "w", encoding="utf-8") as f:
    json.dump(CROATIAN_MARKERS, f, ensure_ascii=False, indent=2)

PR test bilingual: 6,807 rows

Marker preservation rate, PR test set (full corpus):
----------------------------------------------------------------------
  should_statements           720/780      92.3%
  discounting_positive       1409/2369     59.5%
  black_and_white            1007/2137     47.1%
  mental_filtering           1254/2686     46.7%
  fortune_telling            1161/3031     38.3%
  catastrophizing            1011/2677     37.8%
  overgeneralization         1362/3633     37.5%
  mind_reading                502/1729     29.0%
  labeling                    499/2081     24.0%
  personalization              74/2270      3.3%


## 8. Manual inspection helper for refining marker dictionaries

If Section 7 shows a surprisingly low preservation rate for a category, use this cell to inspect actual translations and decide whether (a) the regex is missing real Croatian markers (extend the dictionary in Section 6 and re-run), or (b) the markers genuinely are not in the translations (this is a real finding to report in Chapter 4.4).

Change `INSPECT_CATEGORY` to the category in question.

In [18]:
INSPECT_CATEGORY = "catastrophizing"  # ← change this
N_TO_SHOW = 15
SHOW_MISSES_ONLY = True   # set False to see all

pat = per_cat_compiled[INSPECT_CATEGORY]
sub = pr_test[pr_test[f"marked_{INSPECT_CATEGORY}"] > 0].copy()
sub["match"] = sub["thought_hr"].astype(str).apply(lambda x: bool(pat.search(x)))

show = sub[~sub["match"]] if SHOW_MISSES_ONLY else sub
show = show.sample(min(N_TO_SHOW, len(show)), random_state=RANDOM_SEED) if len(show) else show

header = "MISSED" if SHOW_MISSES_ONLY else "ALL"
print(f"{header} thoughts for category '{INSPECT_CATEGORY}':")
for _, r in show.iterrows():
    print(f"\nEN: {r['thought']}")
    print(f"HR: {r['thought_hr']}")

MISSED thoughts for category 'catastrophizing':

EN: I enjoy crosswords, but I am so stupid I know I am horrible at them.
HR: Volim križaljke, ali sam toliko glup da znam da sam užasan u njima.

EN: My dog misbehaved on our walk yesterday. I'm a terrible pet owner.
HR: Moj pas se jučer loše ponašao u šetnji. Ja sam užasan vlasnik kućnih ljubimaca.

EN: One day my dogs will get angry and kill my children.
HR: Jednog će dana moji psi naljutiti se i ubiti moju djecu.

EN: I've two cats which means that they will definitely get sick and die at some point.
HR: Imam dvije mačke što znači da će se sigurno razboljeti i uginuti u nekom trenutku.

EN: I had lunch with my friend Will Ferrell but he did not pay attention to me. He must not like me.
HR: Ručao sam s prijateljem Willom Ferrellom, ali nije obraćao pažnju na mene. Sigurno mu se ne sviđam.

EN: My scooter would not start this morning. I bet it will cost me thousands to fix.
HR: Moj skuter nije htio upaliti jutros. Kladim se da će me pop

# Part 3 — Manual spot-check sampler

Produce a 30-record CSV stratified across PR categories (3 per category), with empty columns for a 1-5 quality score and free-text notes. Reading and scoring is done offline; the filled CSV gets read back in Section 11 for an aggregate score.

In [19]:
spot_rows = []
for cat in PR_CATEGORIES:
    sub = pr_test[pr_test[f"marked_{cat}"] > 0]
    if len(sub) == 0:
        continue
    sample = sub.sample(min(3, len(sub)), random_state=RANDOM_SEED)
    for _, r in sample.iterrows():
        spot_rows.append({
            "category": cat,
            "en": r["thought"],
            "hr": r["thought_hr"],
            "fluency_1_5": "",     # 1=unintelligible, 5=native-like
            "adequacy_1_5": "",    # 1=meaning lost, 5=fully preserved
            "marker_preserved": "", # yes / no / partial
            "notes": "",
        })

spot_df = pd.DataFrame(spot_rows)
spot_out = AUDIT_DIR / "manual_spotcheck_blank.csv"
spot_df.to_csv(spot_out, index=False)
print(f"Manual spot-check sampler: {len(spot_df)} records written to")
print(f"  {spot_out.resolve()}")
print("\nFill in fluency_1_5, adequacy_1_5, marker_preserved, and notes,")
print("then save as manual_spotcheck_filled.csv and run Section 11.")

Manual spot-check sampler: 30 records written to
  /Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/processed/translation_audit/manual_spotcheck_blank.csv

Fill in fluency_1_5, adequacy_1_5, marker_preserved, and notes,
then save as manual_spotcheck_filled.csv and run Section 11.


## 11. Aggregate scoring (run after filling the spot-check)

Reads the filled spot-check CSV and prints summary statistics. Skip this section until the manual scoring is done.

In [20]:
FILLED_PATH = AUDIT_DIR / "manual_spotcheck_filled.csv"

if not FILLED_PATH.exists():
    print(f"No filled file at {FILLED_PATH}")
    print("Skipping — fill in manual_spotcheck_blank.csv and rename to manual_spotcheck_filled.csv")
else:
    filled = pd.read_csv(FILLED_PATH)
    print(f"Loaded {len(filled)} filled spot-check records\n")

    for col in ["fluency_1_5", "adequacy_1_5"]:
        vals = pd.to_numeric(filled[col], errors="coerce").dropna()
        if len(vals):
            print(f"  {col:<20}  mean={vals.mean():.2f}  median={vals.median():.1f}  n={len(vals)}")

    if "marker_preserved" in filled.columns:
        counts = filled["marker_preserved"].astype(str).str.lower().value_counts()
        print("\n  marker_preserved breakdown:")
        for k, v in counts.items():
            print(f"    {k:<10}  {v}")

    print("\nPer-category mean adequacy:")
    by_cat = filled.copy()
    by_cat["adequacy_num"] = pd.to_numeric(by_cat["adequacy_1_5"], errors="coerce")
    grouped = by_cat.groupby("category")["adequacy_num"].mean().sort_values()
    for cat, val in grouped.items():
        print(f"  {cat:<25}  {val:.2f}")

No filled file at processed/translation_audit/manual_spotcheck_filled.csv
Skipping — fill in manual_spotcheck_blank.csv and rename to manual_spotcheck_filled.csv


## 12. Final summary

Pulls together the audit numbers. This summary is what goes into Chapter 3.4 of the thesis.

In [21]:
print("=" * 70)
print("PHASE F — TRANSLATION QUALITY AUDIT SUMMARY")
print("=" * 70)

print("\n1. Back-translation BLEU / chrF, per dataset:")
for _, r in per_dataset_bleu.iterrows():
    print(f"   {r['dataset']:<5}  n={r['n']:>3}   BLEU={r['bleu']:>5.2f}   chrF={r['chrf']:>5.2f}")

print("\n2. Back-translation BLEU / chrF, per PR distortion category:")
for _, r in per_category_bleu.iterrows():
    print(f"   {r['category']:<25}  n={r['n']:>3}   BLEU={r['bleu']:>5.2f}   chrF={r['chrf']:>5.2f}")

print("\n3. Marker preservation rate, per PR distortion category (full test set):")
for _, r in marker_df.iterrows():
    pct = f"{100*r['preservation_rate']:.1f}%" if pd.notna(r['preservation_rate']) else "n/a"
    print(f"   {r['category']:<25}  n={r['n']:>5}   {pct:>7}")

print("\n4. Files written to processed/translation_audit/:")
for f in sorted(AUDIT_DIR.glob("*.csv")):
    print(f"   {f.name}")

PHASE F — TRANSLATION QUALITY AUDIT SUMMARY

1. Back-translation BLEU / chrF, per dataset:
   TQA    n=200   BLEU=49.07   chrF=72.18
   PR     n=200   BLEU=57.21   chrF=75.42
   CR     n=200   BLEU=52.21   chrF=74.94

2. Back-translation BLEU / chrF, per PR distortion category:
   catastrophizing            n= 20   BLEU=55.45   chrF=75.23
   discounting_positive       n= 20   BLEU=62.09   chrF=79.25
   labeling                   n= 20   BLEU=49.29   chrF=71.49
   mental_filtering           n= 20   BLEU=58.59   chrF=75.05
   mind_reading               n= 20   BLEU=64.70   chrF=79.96
   fortune_telling            n= 20   BLEU=50.99   chrF=69.79
   overgeneralization         n= 20   BLEU=57.10   chrF=76.37
   personalization            n= 20   BLEU=48.58   chrF=70.18
   black_and_white            n= 20   BLEU=63.35   chrF=80.16
   should_statements          n= 20   BLEU=60.68   chrF=77.70

3. Marker preservation rate, per PR distortion category (full test set):
   should_statements       